In [1]:
import warnings
warnings.filterwarnings(action='ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns
import sklearn
from sklearn import datasets
sklearn.set_config(display='text')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, RobustScaler, MinMaxScaler, Normalizer
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score

## 위스콘신 암 데이터를 불러와서 스케일링 한다.

위스콘신 암 데이터를 불러온다.

In [2]:
raw_data = datasets.load_breast_cancer()
print(raw_data)

{'data': array([[1.799e+01, 1.038e+01, 1.228e+02, ..., 2.654e-01, 4.601e-01,
        1.189e-01],
       [2.057e+01, 1.777e+01, 1.329e+02, ..., 1.860e-01, 2.750e-01,
        8.902e-02],
       [1.969e+01, 2.125e+01, 1.300e+02, ..., 2.430e-01, 3.613e-01,
        8.758e-02],
       ...,
       [1.660e+01, 2.808e+01, 1.083e+02, ..., 1.418e-01, 2.218e-01,
        7.820e-02],
       [2.060e+01, 2.933e+01, 1.401e+02, ..., 2.650e-01, 4.087e-01,
        1.240e-01],
       [7.760e+00, 2.454e+01, 4.792e+01, ..., 0.000e+00, 2.871e-01,
        7.039e-02]], shape=(569, 30)), 'target': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0,
       1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0,
       1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0,

위스콘신 암 데이터에서 피처와 피처에 따른 레이블을 얻어온다.

In [3]:
xData = raw_data.data # 피처
yData = raw_data.target # 피처에 따른 레이블
print(xData.shape, yData.shape)

(569, 30) (569,)


학습 데이터와 테스트 데이터로 분할한다.

In [4]:
x_train, x_test, y_train, y_test = train_test_split(xData, yData, random_state=0)
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)

(426, 30) (426,) (143, 30) (143,)


데이터를 스케일링(표준화) 한다.  
학습 데이터는 스케일링 실행 후 적용하고 텍스트 데이터는 학습 데이터로 스케일링된 결과에 적용만 시킨다.

In [5]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

## 양성, 악성 여부를 예측하는 로지스틱 회귀 모델을 생성하고 학습시킨다.

로지스틱 회귀 알고리즘을 하용하기 위해 LogisticRegression을 import 한다.

In [6]:
from sklearn.linear_model import LogisticRegression

로지스틱 회귀 모델은 `penalty 속성`으로 제약 방식을 지정해서 모델을 만드는데 `sklearn 1.10.0` 버전 부터는 `완전히 제거`되어 사용할 수 없게된다.  
`sklearn 1.10.0` 버전 부터는 penalty 대신 `l1_ratio` 속성과 `C` 속성을 조합하여 규제의 종류를 설정해서 모델을 만들어야 한다.  
`l1_ratio` 속성에 `0`을 사용하면 `릿지(L2 규제)`를 사용하고 `1`을 사용하면 `라쏘(L1 규제)`를 사용한다. `비율`을 사용하면 `L1 + L2 혼합 규제`를 사용한다.  
`C`는 `규제 강도의 역수`로, 값이 작을수록 규제가 강해지고, 값이 클수록 규제가 약해진다.  
L1, L2 제약을 모두 지원할 수 있도록 `solver` 속성값을 `saga`로 지정한다.  
`model = LogisticRegression(penalty='l1', solver='saga')` # 버전 1.9 까지 로지스틱 회귀 모델을 만든다.  
`model = LogisticRegression(l1_ratio=0.0, C=0.1, solver='saga')` # 버전 1.10 까지 로지스틱 회귀 모델을 만든다.

In [9]:
# model = LogisticRegression(l1_ratio=0.0, C=0.1, solver='saga')
# model.fit(x_train, y_train)
model = LogisticRegression(l1_ratio=0.0, C=0.1, solver='saga').fit(x_train, y_train)

가중치(회귀 계수)와 바이어스(상수항)를 출력한다.  
선형 회귀 모델에서 coef_ 속성으로 가중치(피처의 개수)와 intercept_ 속성으로 바이어스(무조건 1개)를 얻어온다.

In [11]:
print('가중치:\n', model.coef_, sep='')
print('바이어스:\n', model.intercept_, sep='')

가중치:
[[-0.35119661 -0.39947514 -0.34425981 -0.3355599  -0.18267041 -0.03684577
  -0.3128862  -0.4172135  -0.20460524  0.1931644  -0.50756333 -0.00449978
  -0.40931249 -0.36479951  0.10124564  0.25040541  0.09137784 -0.11091285
   0.11193166  0.2698863  -0.48039854 -0.49332484 -0.45292414 -0.42340931
  -0.32675249 -0.16645165 -0.37931112 -0.49689415 -0.37277991 -0.17354561]]
바이어스:
[0.46405864]


## 학습된 모델로 테스트 데이터를 예측한다.

In [15]:
predict = model.predict(x_test)
print(predict)

[0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 0 0 0 0 0 1 1 0 1 1 0 1 0 1 0 1 0 1 0 1
 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 1 1 1 1 1 1 0 0 0 1 1 0 1 0 0 0 1 1 0 1 0
 0 1 1 1 1 1 0 0 0 1 0 1 1 1 0 0 1 1 0 0 1 1 0 1 1 1 1 1 1 1 0 1 0 1 1 1 1
 0 0 1 1 1 1 1 1 1 1 1 0 1 0 1 1 1 1 1 0 1 1 1 1 1 1 1 0 1 1 1 0]


In [16]:
predict_proba = model.predict_proba(x_test)
print(predict_proba)

[[9.50972596e-01 4.90274039e-02]
 [8.87143645e-02 9.11285635e-01]
 [1.06363432e-02 9.89363657e-01]
 [3.96608880e-02 9.60339112e-01]
 [4.11625162e-03 9.95883748e-01]
 [2.61338332e-02 9.73866167e-01]
 [6.76595646e-03 9.93234044e-01]
 [1.33102831e-02 9.86689717e-01]
 [1.37318677e-03 9.98626813e-01]
 [5.49531239e-04 9.99450469e-01]
 [3.06175353e-01 6.93824647e-01]
 [1.60158664e-01 8.39841336e-01]
 [8.01991425e-04 9.99198009e-01]
 [4.18871420e-01 5.81128580e-01]
 [4.25566819e-01 5.74433181e-01]
 [9.25691264e-01 7.43087361e-02]
 [2.27864817e-02 9.77213518e-01]
 [9.99111619e-01 8.88380990e-04]
 [9.95245166e-01 4.75483431e-03]
 [9.99966467e-01 3.35330882e-05]
 [9.54689392e-01 4.53106082e-02]
 [9.26235094e-01 7.37649057e-02]
 [6.98563122e-02 9.30143688e-01]
 [1.14114792e-02 9.88588521e-01]
 [9.89882539e-01 1.01174610e-02]
 [7.16974190e-03 9.92830258e-01]
 [1.17908637e-03 9.98820914e-01]
 [9.54974287e-01 4.50257134e-02]
 [1.52255577e-02 9.84774442e-01]
 [9.99344597e-01 6.55402955e-04]
 [1.452506

## 모델을 평가한다.

In [19]:
classification = classification_report(y_test, predict, target_names=raw_data.target_names)
# classification = classification_report(y_test, predict, target_names=['악성', '양성'])
print(classification)

              precision    recall  f1-score   support

   malignant       0.98      0.92      0.95        53
      benign       0.96      0.99      0.97        90

    accuracy                           0.97       143
   macro avg       0.97      0.96      0.96       143
weighted avg       0.97      0.97      0.96       143

